In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

In [2]:
# 
X_train = pd.read_csv('X_train.csv')
X_test  = pd.read_csv('X_test.csv')
y_test = pd.read_csv('y_test.csv')
y_train = pd.read_csv('y_train.csv')

In [3]:
ord_map = {
    '1-12': 1, '13-24': 2, '25++': 3,
    'oneyear': 1, 'twoyear': 2, 'threeyear': 3,
    '1year': 1, '2year': 2, '3year': 3  # just in case
}

In [4]:
X_train['tenure_ord'] = X_train['tenure'].map(ord_map)
X_test['tenure_ord']  = X_test['tenure'].map(ord_map)

In [5]:
X_train = X_train.drop(columns=['tenure'])
X_test  = X_test.drop(columns=['tenure'])

In [6]:
X_train.head(10)

,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,is_married,tenure_ord
0,Fiber optic,Yes,Yes,Yes,Yes,No,No,Two year,No,Credit card (automatic),94.55,1,3
1,DSL,No,No,Yes,Yes,No,No,Month-to-month,No,Electronic check,35.75,0,3
2,Fiber optic,No,Yes,Yes,Yes,No,No,Two year,No,Credit card (automatic),90.20,1,3
3,Fiber optic,No,Yes,No,No,No,Yes,Month-to-month,No,Electronic check,84.30,0,1
4,DSL,Yes,No,No,No,Yes,No,Month-to-month,No,Bank transfer (automatic),40.65,1,3
5,Fiber optic,Yes,Yes,Yes,No,No,Yes,One year,No,Bank transfer (automatic),97.00,1,3
6,DSL,No,No,Yes,Yes,Yes,No,Month-to-month,No,Bank transfer (automatic),64.30,0,1
7,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Mailed check,93.20,0,2
8,DSL,No,No,Yes,Yes,No,No,Month-to-month,No,Mailed check,54.35,1,3
9,DSL,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,47.95,0,1


In [7]:
print(X_train['tenure_ord'].value_counts())

tenure_ord
3    3083
1    1724
2     818
Name: count, dtype: int64


In [8]:
def binary_encode(df, cols):
    for col in cols:
        df[col] = df[col].astype(str).str.strip().str.lower().map({
            'yes': 1, 'no': 0, '1': 1, '0': 0
        }).astype(int)
    return df

In [9]:
X_train = binary_encode(X_train, ["is_married", "PaperlessBilling"])
X_test  = binary_encode(X_test, ["is_married", "PaperlessBilling"])

In [10]:
print("is_married train:", X_train["is_married"].value_counts())
print("PaperlessBilling train:", X_train["PaperlessBilling"].value_counts())
print("Churn train:", y_train["Churn"].value_counts())

is_married train: is_married
1    3460
0    2165
Name: count, dtype: int64
PaperlessBilling train: PaperlessBilling
1    3349
0    2276
Name: count, dtype: int64
Churn train: Churn
0    4130
1    1495
Name: count, dtype: int64


In [11]:
X_train.head()

,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,is_married,tenure_ord
0,Fiber optic,Yes,Yes,Yes,Yes,No,No,Two year,0,Credit card (automatic),94.55,1,3
1,DSL,No,No,Yes,Yes,No,No,Month-to-month,0,Electronic check,35.75,0,3
2,Fiber optic,No,Yes,Yes,Yes,No,No,Two year,0,Credit card (automatic),90.20,1,3
3,Fiber optic,No,Yes,No,No,No,Yes,Month-to-month,0,Electronic check,84.30,0,1
4,DSL,Yes,No,No,No,Yes,No,Month-to-month,0,Bank transfer (automatic),40.65,1,3


In [12]:
y_train.head()

,Churn
0,0
1,0
2,0
3,0
4,0


In [13]:
ohe_cols = [
    "InternetService","OnlineSecurity","OnlineBackup","DeviceProtection",
    "TechSupport","StreamingTV","StreamingMovies","Contract","PaymentMethod"
]

In [14]:

# OHE pakai get_dummies (fit di train, align ke test)
X_train_ohe = pd.get_dummies(X_train, columns=ohe_cols)
X_test_ohe  = pd.get_dummies(X_test,  columns=ohe_cols)

In [15]:
# Pastikan kolom sama urutan dan jumlahnya
X_test_ohe = X_test_ohe.reindex(columns=X_train_ohe.columns, fill_value=0)


In [16]:
X_train_ohe.head()

,PaperlessBilling,MonthlyCharges,is_married,tenure_ord,InternetService_DSL,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,...,StreamingMovies_No,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,94.55,1,3,False,True,False,False,False,True,...,True,False,False,False,False,True,False,True,False,False
1,0,35.75,0,3,True,False,False,True,False,False,...,True,False,False,True,False,False,False,False,True,False
2,0,90.20,1,3,False,True,False,True,False,False,...,True,False,False,False,False,True,False,True,False,False
3,0,84.30,0,1,False,True,False,True,False,False,...,False,False,True,True,False,False,False,False,True,False
4,0,40.65,1,3,True,False,False,False,False,True,...,True,False,False,True,False,False,True,False,False,False


In [17]:
# Simpan
X_train_ohe.to_csv("X_train_ohe.csv", index=False)
X_test_ohe.to_csv("X_test_ohe.csv", index=False)

In [18]:

# Load hasil OHE
X_train_ohe = pd.read_csv("X_train_ohe.csv")
X_test_ohe  = pd.read_csv("X_test_ohe.csv")



In [19]:
# Convert semua kolom bool ke int
X_train_ohe = X_train_ohe.astype(int)
X_test_ohe  = X_test_ohe.astype(int)


In [21]:
X_train_ohe.head()

,PaperlessBilling,MonthlyCharges,is_married,tenure_ord,InternetService_DSL,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,...,StreamingMovies_No,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,94,1,3,0,1,0,0,0,1,...,1,0,0,0,0,1,0,1,0,0
1,0,35,0,3,1,0,0,1,0,0,...,1,0,0,1,0,0,0,0,1,0
2,0,90,1,3,0,1,0,1,0,0,...,1,0,0,0,0,1,0,1,0,0
3,0,84,0,1,0,1,0,1,0,0,...,0,0,1,1,0,0,0,0,1,0
4,0,40,1,3,1,0,0,0,0,1,...,1,0,0,1,0,0,1,0,0,0


In [22]:
X_train_ohe.to_csv("X_train_ohe_int.csv", index=False)
X_test_ohe.to_csv("X_test_ohe_int.csv", index=False)

In [24]:
X_train_ohe.columns

Index(['PaperlessBilling', 'MonthlyCharges', 'is_married', 'tenure_ord',
       'InternetService_DSL', 'InternetService_Fiber optic',
       'InternetService_No', 'OnlineSecurity_No',
       'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
       'OnlineBackup_No', 'OnlineBackup_No internet service',
       'OnlineBackup_Yes', 'DeviceProtection_No',
       'DeviceProtection_No internet service', 'DeviceProtection_Yes',
       'TechSupport_No', 'TechSupport_No internet service', 'TechSupport_Yes',
       'StreamingTV_No', 'StreamingTV_No internet service', 'StreamingTV_Yes',
       'StreamingMovies_No', 'StreamingMovies_No internet service',
       'StreamingMovies_Yes', 'Contract_Month-to-month', 'Contract_One year',
       'Contract_Two year', 'PaymentMethod_Bank transfer (automatic)',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check'],
      dtype='object')